In [1]:
# """
# GraphCast Inference Pipeline
# =============================
# Model: GraphCast_operational  (0.25 deg, 13 pressure levels)
# Input: ERA5 data downloaded via CDS API (two consecutive 6-hourly time steps)
# Output: N-step rollout of predictions (each step = 6 hours ahead)

# WHAT YOU NEED TO DOWNLOAD
# --------------------------
# 1. Model checkpoint (from Google Cloud Bucket gs://dm_graphcast):
#      graphcast/params/GraphCast_operational - ERA5-HRES 0.25deg - pressure levels 13 - mesh 2to6 - precipitation 0 - 2039vars.npz
#    (or choose the ERA5 variant if you want precipitation input)

# 2. Normalization statistics (same bucket):
#      graphcast/stats/diffs_stddev_by_level.nc
#      graphcast/stats/mean_by_level.nc
#      graphcast/stats/stddev_by_level.nc

# 3. ERA5 data via CDS API (see download_era5_inputs() below):
#    - Two consecutive 6-hourly time steps  (e.g. 00Z and 06Z on your init date)
#    - Surface variables + pressure-level variables listed in ERA5_VARIABLES

# PIPELINE FLOW
# -------------
#   download_era5_inputs()          # Pull ERA5 from CDS as NetCDF files
#       │
#   build_era5_xarray_batch()       # Merge surface + pressure-level files into
#       │                           # a single xarray.Dataset with the exact
#       │                           # dimensions GraphCast expects
#       │                           #   (batch, time, lat, lon) or
#       │                           #   (batch, time, level, lat, lon)
#       │
#   load_checkpoint()               # Read .npz checkpoint → params, model_config,
#       │                           # task_config
#       │
#   load_normalization_stats()      # Read three .nc stat files
#       │
#   build_run_forward_jitted()      # Wrap GraphCast with normalization + bfloat16
#       │                           # cast + autoregressive predictor, then jit
#       │
#   extract_inputs_targets_forcings()  # Split batch into inputs (t-6h, t0),
#       │                              # targets_template (NaN placeholder),
#       │                              # forcings (solar radiation)
#       │
#   run_rollout()                   # rollout.chunked_prediction → predictions
#       │
#   plot_forecast() / animate_forecast()   # Visualisation helpers
# """

# GraphCast Inference Pipeline

**Model:** `GraphCast_operational` — 0.25° resolution, 13 pressure levels  
**Input:** ERA5 reanalysis data downloaded via CDS API  
**Output:** N-step autoregressive rollout, each step = 6 hours ahead

---

## Files to Download

### 1. Model Checkpoint — Google Cloud Bucket `gs://dm_graphcast/`

```
graphcast/params/GraphCast_operational - ERA5-HRES 0.25deg - pressure levels 13 - mesh 2to6 - precipitation 0 - 2039vars.npz
```

> **Which checkpoint to use?**
> | Checkpoint | Resolution | Levels | Input precip? | Notes |
> |---|---|---|---|---|
> | `GraphCast` | 0.25° | 37 | Yes (ERA5) | Full model from the paper |
> | `GraphCast_small` | 1° | 13 | Yes (ERA5) | Cheaper; lower accuracy |
> | `GraphCast_operational` | 0.25° | 13 | No | Pre-trained ERA5, fine-tuned on HRES. Best for operational use. |
>
> This pipeline targets `GraphCast_operational`. It does **not** require precipitation as input, which simplifies the ERA5 download.

### 2. Normalization Statistics — same bucket

```
graphcast/stats/diffs_stddev_by_level.nc
graphcast/stats/mean_by_level.nc
graphcast/stats/stddev_by_level.nc
```

These are the same three files regardless of which checkpoint you use. They encode the per-variable, per-level mean and standard deviation of the ERA5 training data, and are used to normalize inputs and denormalize outputs inside the model wrapper.

---

## ERA5 Variables Required

### Pressure-level variables
Downloaded at **13 levels**: `[50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]` hPa

| CDS variable name | GraphCast internal name |
|---|---|
| `geopotential` | `geopotential` |
| `specific_humidity` | `specific_humidity` |
| `temperature` | `temperature` |
| `u_component_of_wind` | `u_component_of_wind` |
| `v_component_of_wind` | `v_component_of_wind` |
| `vertical_velocity` | `vertical_velocity` |

### Surface (single-level) variables

| CDS variable name | GraphCast internal name |
|---|---|
| `2m_temperature` | `2m_temperature` |
| `mean_sea_level_pressure` | `mean_sea_level_pressure` |
| `10m_u_component_of_wind` | `10m_u_component_of_wind` |
| `10m_v_component_of_wind` | `10m_v_component_of_wind` |

> `total_precipitation` is **not needed** for `GraphCast_operational`. Only add it if you switch to the full `GraphCast` ERA5 variant.

### Solar radiation (forcing)

| CDS variable name | GraphCast internal name |
|---|---|
| `toa_incident_solar_radiation` | `toa_incident_solar_radiation` |

This is downloaded separately for the **future rollout time steps only** (see timeline below). If you skip it, GraphCast computes solar radiation analytically from lat/lon/time — accurate enough for most uses.

### Static variables (time-invariant)

| CDS variable name | GraphCast internal name |
|---|---|
| `geopotential` (at surface) | `geopotential_at_surface` / orography |
| `land_sea_mask` | `land_sea_mask` |

Only one time snapshot is needed. These fields never change.

---

## Input Time Step Logic

GraphCast requires **exactly two atmospheric snapshots** as input. These are not a time series — they are two 6-hourly analysis fields that give the model both the current state and its recent tendency.

```
Given:   init_time = 2024-01-01 00Z,  rollout_steps = 40

ERA5 download:
  Atmospheric state (2 steps):
    2023-12-31 12Z   <- init_time - 12h   (first input,  GraphCast t = -6h)
    2023-12-31 18Z   <- init_time -  6h   (second input, GraphCast t =  0h)

  Solar radiation forcing (rollout_steps steps):
    2024-01-01 06Z   <- t = +6h   (first prediction step forcing)
    2024-01-01 12Z   <- t = +12h
    ...
    2024-01-11 00Z   <- t = +240h = +40 × 6h

  Static fields (1 snapshot):
    any single time  <- orography, land-sea mask
```

> **Important convention:** GraphCast's internal t=0 is the **second** input time step (i.e. `init_time - 6h`), not `init_time` itself. All timedelta coordinates in the xarray Dataset are expressed relative to this point.

---

## Pipeline Steps

### Step 1 — `download_era5_inputs()`

Calls the CDS API to download three NetCDF files into a local directory:

- `era5_pressure_levels.nc` — 3D atmospheric state at 2 time steps and 13 pressure levels
- `era5_surface.nc` — 2D surface state at 2 time steps
- `era5_static.nc` — orography and land-sea mask (1 snapshot)
- `era5_solar_forcing.nc` — top-of-atmosphere solar radiation at all future forcing steps (optional)

All fields are requested at **0.25° global resolution**.

---

### Step 2 — `build_era5_xarray_batch()`

Merges the downloaded files into a single `xarray.Dataset` with the exact structure GraphCast expects. Key transformations:

1. **Rename variables** — CDS short names (`z`, `t`, `u`, `v`, `w`, `q`, `t2m`, `msl`, ...) are mapped to GraphCast's internal variable names.
2. **Rename coordinates** — `latitude → lat`, `longitude → lon`, `pressure_level → level`, `valid_time → time`.
3. **Flip latitude** — CDS returns data south-up (-90 … 90). GraphCast expects north-down (90 … -90).
4. **Convert timestamps to timedeltas** — absolute datetimes are converted to `np.timedelta64` relative to `t=0` (`init_time - 6h`):
   - `init_time - 12h` → `timedelta(-6h)`
   - `init_time -  6h` → `timedelta(  0h)`
   - `init_time +  6h` → `timedelta(+6h)` ← first forcing step
5. **Strip time from static fields** — orography and land-sea mask have no time dimension.
6. **Merge** — all datasets are merged into one `xarray.Dataset`.
7. **Add batch dimension** — `expand_dims("batch")` adds a size-1 batch axis. GraphCast always expects this even at inference.

Final dataset dimensions:
```
Surface variables :  (batch=1, time=2,              lat=721, lon=1440)
Pressure variables:  (batch=1, time=2, level=13,    lat=721, lon=1440)
Solar forcing     :  (batch=1, time=rollout_steps,  lat=721, lon=1440)
Static variables  :                                  lat=721, lon=1440
```

---

### Step 3 — `load_checkpoint()`

Reads the `.npz` checkpoint file and unpacks:

- `params` — the model weights (a nested dict of JAX arrays)
- `state` — always empty `{}` at inference (GraphCast is stateless)
- `model_config` — architecture hyperparameters (mesh size, latent size, GNN steps, resolution)
- `task_config` — data contract (which variables are inputs/targets/forcings, pressure levels, time step duration)

---

### Step 4 — `load_normalization_stats()`

Loads the three `.nc` files into `xarray.Dataset` objects:

- `mean_by_level` — per-variable, per-level mean across the ERA5 training set
- `stddev_by_level` — per-variable, per-level standard deviation
- `diffs_stddev_by_level` — standard deviation of 6-hourly differences (used to scale the residual outputs)

These are used in Step 5 inside the `InputsAndResiduals` wrapper.

---

### Step 5 — `build_run_forward_jitted()`

Constructs the full model by stacking four wrappers around the core GNN, then JIT-compiles the single-step forward pass.

**Wrapper stack (inner → outer):**

```
GraphCast
    The core graph neural network. Operates on an icosahedral mesh
    of ~40,000 nodes. Runs encoder → processor (GNN message passing)
    → decoder to produce one 6-hour forecast step as residuals.

Bfloat16Cast
    Converts inputs to bfloat16 before the GNN and converts outputs
    back to float32. Reduces memory and speeds up computation on
    TPUs/GPUs that natively support bfloat16.

InputsAndResiduals
    - On the way IN:  normalizes inputs using mean_by_level and stddev_by_level.
    - On the way OUT: the model predicts residuals (changes), not absolute values.
      Denormalizes residuals using diffs_stddev_by_level and adds them to the
      previous state to produce the next absolute atmospheric state.

autoregressive.Predictor
    Wraps the one-step model to produce multi-step trajectories.
    At each rollout step it feeds the model's own previous prediction
    back as the new input (autoregressive loop).
    gradient_checkpointing=False at inference (only needed for training).
```

The JIT-compiled function has the signature:
```python
run_forward_jitted(rng, inputs, targets_template, forcings) -> predictions
```

---

### Step 6 — `prepare_inputs()`

Calls `data_utils.extract_inputs_targets_forcings()` to split the merged batch into the three tensors the model needs:

| Tensor | Time steps | Content |
|---|---|---|
| `inputs` | `[-6h, 0h]` | All atmospheric variables at the two input snapshots |
| `targets_template` | `[+6h, ..., +N×6h]` | NaN-filled placeholder — defines output shape only |
| `forcings` | `[+6h, ..., +N×6h]` | `toa_incident_solar_radiation` at each future step |

**Why NaN for targets?** At inference there is no ground truth. The template just tells GraphCast how many steps to produce and what shape each output should be. At training time you would fill this with real ERA5 values to compute loss.

**Why does forcings only contain solar radiation?** Solar radiation is the only variable in `task_config.forcing_variables`. It is known exactly for any future time and location (it is deterministic), so it can be provided to the model even when forecasting the future. All other variables are either predicted autoregressively or are static.

---

### Step 7 — `run_rollout()`

Calls `rollout.chunked_prediction()` which loops over single-step forward calls in Python:

```
step 1:  inputs=[t-6h, t0]        → predicts t+6h
step 2:  inputs=[t0,   t+6h]      → predicts t+12h
step 3:  inputs=[t+6h, t+12h]     → predicts t+18h
...
step N:  inputs=[t+(N-1)*6h, t+N*6h-6h]  → predicts t+N*6h
```

At each step the most recent prediction is fed back as the new input. The JAX function is compiled only once (on the first step); all subsequent steps reuse the compiled code and run fast.

**Output:** `xarray.Dataset` with all predicted variables across all rollout steps.

---

### Step 8 — Visualisation

Three helper functions are provided:

| Function | Description |
|---|---|
| `plot_forecast()` | Single variable, single step — global map with colorbar |
| `plot_multi_variable_snapshot()` | Multiple variables side-by-side at one step |
| `animate_forecast()` | Animated HTML widget cycling through all forecast steps (Jupyter) |

All plots display the valid time (absolute datetime) derived from `init_time + lead_hours` in the title.

---

## Complete Data Flow Diagram

```
CDS API
  │
  ├── era5_pressure_levels.nc   [2 time steps × 6 vars × 13 levels × 721 × 1440]
  ├── era5_surface.nc           [2 time steps × 4 vars × 721 × 1440]
  ├── era5_static.nc            [1 snapshot  × 2 vars × 721 × 1440]
  └── era5_solar_forcing.nc     [N time steps × 1 var × 721 × 1440]
        │
        ▼
  build_era5_xarray_batch()
    rename vars, rename coords, flip lat,
    convert timestamps → timedeltas, merge, add batch dim
        │
        ▼
  xarray.Dataset
    dims: (batch=1, time=2+N, lat=721, lon=1440[, level=13])
        │
        ├──────────────────────────────────────────────────┐
        │                                                  │
        ▼                                                  ▼
  load_checkpoint()                            load_normalization_stats()
  params, model_config, task_config            diffs_stddev, mean, stddev
        │                                                  │
        └──────────────┬───────────────────────────────────┘
                       │
                       ▼
  build_run_forward_jitted()
    GraphCast → Bfloat16Cast → InputsAndResiduals → autoregressive.Predictor
    jax.jit(run_forward.apply)
                       │
                       ▼
  prepare_inputs()
    extract_inputs_targets_forcings()
    inputs [t-6h, t0] | targets_template [NaN] | forcings [solar t+6h…t+N]
                       │
                       ▼
  run_rollout()
    rollout.chunked_prediction()
    autoregressive loop: step 1 … step N
                       │
                       ▼
  predictions  xarray.Dataset
    (batch=1, time=N, lat=721, lon=1440[, level=13])
                       │
                       ▼
  plot_forecast() / animate_forecast() / plot_multi_variable_snapshot()
```


In [2]:


# stdlib
import dataclasses
import datetime
import functools
import math
from pathlib import Path
from typing import Optional

# third-party
import cdsapi
import haiku as hk
import jax
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import animation
from IPython.display import HTML
import numpy as np
import xarray

# graphcast
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout


In [3]:


# =============================================================================
# 0.  CONSTANTS
# =============================================================================

PRESSURE_LEVELS_13 = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]

# Pressure-level variables (CDS long names used in the API request)
ERA5_PRESSURE_VARS_CDS = [
    "geopotential",
    "specific_humidity",
    "temperature",
    "u_component_of_wind",
    "v_component_of_wind",
    "vertical_velocity",
]

# Surface variables for the two input time steps.
# NOTE: total_precipitation is NOT needed for GraphCast_operational
# (it does not use precipitation as input).
# Add it back only if using the full GraphCast ERA5 variant (37 levels).
ERA5_SURFACE_VARS_CDS = [
    "2m_temperature",
    "mean_sea_level_pressure",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

# Solar radiation -- needed for every FUTURE rollout step as a forcing.
# If use_era5_solar=False, GraphCast computes this analytically.
ERA5_SOLAR_VAR_CDS = "toa_incident_solar_radiation"

# Static fields -- time-invariant; only one snapshot needed.
ERA5_STATIC_VARS_CDS = [
    "geopotential",    # surface orography (z at model surface, not a pressure level)
    "land_sea_mask",
]

# CDS short name (as returned in the NetCDF) -> GraphCast internal variable name
CDS_TO_GRAPHCAST_NAME = {
    # pressure-level
    "z":    "geopotential",
    "q":    "specific_humidity",
    "t":    "temperature",
    "u":    "u_component_of_wind",
    "v":    "v_component_of_wind",
    "w":    "vertical_velocity",
    # surface
    "t2m":  "2m_temperature",
    "msl":  "mean_sea_level_pressure",
    "u10":  "10m_u_component_of_wind",
    "v10":  "10m_v_component_of_wind",
    "tisr": "toa_incident_solar_radiation",
    # "tp":   "total_precipitation_6hr",
    # static
    "lsm":  "land_sea_mask",
}


def download_era5_inputs(
    init_time: datetime.datetime,
    rollout_steps: int,
    output_dir: str | Path = "era5_data",
    use_era5_solar: bool = True,
) -> dict[str, Path]:

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    c = cdsapi.Client()

    t_minus_12h = init_time - datetime.timedelta(hours=12)
    t_minus_6h  = init_time - datetime.timedelta(hours=6)

    dates = sorted({t_minus_12h.strftime("%Y-%m-%d"), t_minus_6h.strftime("%Y-%m-%d")})
    date_range = f"{dates[0]}/{dates[-1]}"
    times = sorted({t_minus_12h.strftime("%H:%M"), t_minus_6h.strftime("%H:%M")})

    def _download(label, path, dataset, request):
        if path.exists():
            print(f"[SKIP] {label} -> {path}")
            return
        print(f"[DOWNLOAD] {label} -> {path}")
        c.retrieve(dataset, request, str(path))
        print(f"[DONE] {label}")

    common = {"product_type": "reanalysis", "area": [90, -180, -90, 180],
              "grid": [0.25, 0.25], "format": "netcdf",
              "date": date_range, "time": times}

    # Pressure-level vars at the 2 input timesteps
    pl_path = output_dir / "era5_pressure_levels.nc"
    _download("Pressure-level variables", pl_path, "reanalysis-era5-pressure-levels", {
        **common,
        "variable":       ERA5_PRESSURE_VARS_CDS,
        "pressure_level": [str(p) for p in PRESSURE_LEVELS_13],
    })

    # Surface vars at the 2 input timesteps
    sl_path = output_dir / "era5_surface.nc"
    _download("Surface variables", sl_path, "reanalysis-era5-single-levels", {
        **common, "variable": ERA5_SURFACE_VARS_CDS,
    })

    # Static vars -- time-invariant, only need one snapshot
    static_path = output_dir / "era5_static.nc"
    _download("Static variables", static_path, "reanalysis-era5-single-levels", {
        "product_type": "reanalysis", "area": [90, -180, -90, 180],
        "grid": [0.25, 0.25], "format": "netcdf",
        "variable": ERA5_STATIC_VARS_CDS,
        "date": t_minus_6h.strftime("%Y-%m-%d"),
        "time": [t_minus_6h.strftime("%H:%M")],
    })

    # Solar at the SAME 2 input timesteps -- same as all other vars.
    # data_utils.add_tisr_var computes solar analytically for future
    # forcing steps during extract_inputs_targets_forcings.
    solar_path = output_dir / "era5_solar.nc"
    if use_era5_solar:
        _download("Solar radiation", solar_path, "reanalysis-era5-single-levels", {
            **common, "variable": [ERA5_SOLAR_VAR_CDS],
        })

    print("ERA5 download complete.")
    return {
        "pressure_levels": pl_path,
        "surface":         sl_path,
        "static":          static_path,
        **({"solar": solar_path} if use_era5_solar else {}),
    }


def build_era5_xarray_batch(
    nc_paths: dict[str, Path],
    init_time: datetime.datetime,
    rollout_steps: int,
) -> xarray.Dataset:

    t_zero = np.datetime64(init_time - datetime.timedelta(hours=12), "ns")
    JUNK   = ["number", "expver"]

    def _load(key):
        return xarray.load_dataset(nc_paths[key], drop_variables=JUNK).compute()

    def _prep(ds):
        ds = ds.rename({k: v for k, v in CDS_TO_GRAPHCAST_NAME.items() if k in ds.data_vars})
        ds = ds.rename({k: v for k, v in {
            "latitude": "lat", "longitude": "lon",
            "pressure_level": "level", "valid_time": "time",
        }.items() if k in ds.dims or k in ds.coords})
        if "time" in ds.coords:
            ds = ds.assign_coords(time=ds["time"].values.astype("datetime64[ns]") - t_zero)
        if "lat" in ds.dims and ds["lat"].values[0] > ds["lat"].values[-1]:
            ds = ds.isel(lat=slice(None, None, -1))
        if "lon" in ds.dims and float(ds["lon"].min()) < 0:
            ds = ds.assign_coords(lon=ds["lon"] % 360).sortby("lon")
        ds = ds.assign_coords(
            lat=ds["lat"].astype(np.float32),
            lon=ds["lon"].astype(np.float32),
        )
        if "level" in ds.coords:
            ds = ds.assign_coords(level=ds["level"].astype(np.int32))
        return ds

    # Load and prep the 2 input timesteps
    ds_pl = _prep(_load("pressure_levels"))
    ds_sl = _prep(_load("surface"))

    # Verify no NaN in real input data
    for label, ds in [("pressure_levels", ds_pl), ("surface", ds_sl)]:
        for var in ds.data_vars:
            n = int(np.isnan(ds[var].values).sum())
            if n > 0:
                raise ValueError(f"NaN in '{var}' ({label}): {n} -- check download")

    # Merge the 2 real input timesteps
    ds = xarray.merge([ds_pl, ds_sl], join="inner")
    assert ds.sizes["time"] == 2, f"Expected 2 input timesteps, got {ds.sizes['time']}"

    # Pad with rollout_steps NaN slots at future timesteps.
    # These slots anchor the time axis so extract_input_target_times
    # internal shift = -6h, correctly placing inputs at [-6h, 0h].
    # All variables are NaN at future slots -- correct because:
    #   - atmo vars: become targets_template (wiped to NaN anyway)
    #   - solar: filled analytically by data_utils.add_tisr_var
    future_times = np.array([
        np.timedelta64((i + 2) * 6, "h") for i in range(rollout_steps)
    ], dtype="timedelta64[ns]")

    # Create NaN-filled future slots matching all current data_vars
    future_data = {
        var: xarray.DataArray(
            np.full(
                tuple(
                    len(future_times) if d == "time" else ds.sizes[d]
                    for d in ds[var].dims
                ),
                np.nan,
                dtype=np.float32,
            ),
            dims=ds[var].dims,
            coords={
                d: (future_times if d == "time" else ds.coords[d])
                for d in ds[var].dims
                if d in ds.coords or d == "time"
            },
        )
        for var in ds.data_vars
    }
    ds_future = xarray.Dataset(future_data)

    # Concatenate real (2 steps) + future NaN (rollout_steps) along time
    ds = xarray.concat([ds, ds_future], dim="time")
    assert ds.sizes["time"] == 2 + rollout_steps, \
        f"Expected {2 + rollout_steps} time steps, got {ds.sizes['time']}"

    # Static -- no time dim
    ds_st = _prep(_load("static")).isel(time=0, drop=True)
    if "geopotential" in ds_st.data_vars:
        ds_st = ds_st.rename({"geopotential": "geopotential_at_surface"})

    # Add batch dim, attach static
    ds = ds.expand_dims("batch", axis=0)
    for var in ds_st.data_vars:
        ds[var] = ds_st[var]

    # Precipitation placeholder (target-only, not an input)
    if "total_precipitation_6hr" not in ds.data_vars:
        ds["total_precipitation_6hr"] = xarray.zeros_like(ds["2m_temperature"])

    # datetime coord (batch, time) required by data_utils.add_derived_vars
    abs_times = (t_zero + ds["time"].values).astype("datetime64[ns]")
    ds = ds.assign_coords(datetime=(("batch", "time"), abs_times[np.newaxis, :]))

    print(f"Batch: {dict(ds.sizes)}")
    print(f"  time : {ds['time'].values}")
    print(f"  vars : {sorted(ds.data_vars)}")
    return ds
    
# =============================================================================
# 3.  STEP 3 -- Load checkpoint
# =============================================================================

def load_checkpoint(ckpt_path: str | Path):
    """Load model checkpoint from the .npz file downloaded from GCS.

    Returns
    -------
    params, state, model_config, task_config
    """
    with open(ckpt_path, "rb") as f:
        ckpt = checkpoint.load(f, graphcast.CheckPoint)

    params       = ckpt.params
    state        = {}   # GraphCast has no mutable state at inference
    model_config = ckpt.model_config
    task_config  = ckpt.task_config

    print("Model description:\n", ckpt.description)
    print("\nmodel_config:", model_config)
    print("task_config :", task_config)
    return params, state, model_config, task_config


# =============================================================================
# 4.  STEP 4 -- Load normalization statistics
# =============================================================================

def load_normalization_stats(stats_dir: str | Path):
    """Load the three normalization .nc files.

    Returns
    -------
    diffs_stddev_by_level, mean_by_level, stddev_by_level
    """
    stats_dir    = Path(stats_dir)
    diffs_stddev = xarray.load_dataset(stats_dir / "diffs_stddev_by_level.nc").compute()
    mean         = xarray.load_dataset(stats_dir / "mean_by_level.nc").compute()
    stddev       = xarray.load_dataset(stats_dir / "stddev_by_level.nc").compute()
    return diffs_stddev, mean, stddev


# =============================================================================
# 5.  STEP 5 -- Build the JIT-compiled forward function
# =============================================================================

def build_run_forward_jitted(
    model_config,
    task_config,
    params,
    state,
    diffs_stddev_by_level,
    mean_by_level,
    stddev_by_level,
):
    """Wrap GraphCast and JIT-compile the single-step forward pass.

    Wrapper stack (inner -> outer):
        GraphCast                core GNN
        Bfloat16Cast             float32 <-> bfloat16 conversion
        InputsAndResiduals       normalise inputs; denormalise residual outputs
        autoregressive.Predictor handles the multi-step rollout loop

    All training-related functions (loss_fn, grads_fn, init) are excluded.

    Returns
    -------
    run_forward_jitted : callable
        run_forward_jitted(rng, inputs, targets_template, forcings)
    """
    def construct_wrapped_graphcast(model_config, task_config):
        predictor = graphcast.GraphCast(model_config, task_config)
        predictor = casting.Bfloat16Cast(predictor)
        predictor = normalization.InputsAndResiduals(
            predictor,
            diffs_stddev_by_level=diffs_stddev_by_level,
            mean_by_level=mean_by_level,
            stddev_by_level=stddev_by_level,
        )
        predictor = autoregressive.Predictor(predictor, gradient_checkpointing=False)
        return predictor

    @hk.transform_with_state
    def run_forward(model_config, task_config, inputs, targets_template, forcings):
        predictor = construct_wrapped_graphcast(model_config, task_config)
        return predictor(inputs, targets_template=targets_template, forcings=forcings)

    def with_configs(fn):
        return functools.partial(fn, model_config=model_config, task_config=task_config)

    def with_params(fn):
        return functools.partial(fn, params=params, state=state)

    def drop_state(fn):
        # GraphCast is stateless; return predictions only, not (predictions, state)
        return lambda **kw: fn(**kw)[0]

    run_forward_jitted = drop_state(
        with_params(jax.jit(with_configs(run_forward.apply)))
    )
    return run_forward_jitted


# =============================================================================
# 6.  STEP 6 -- Prepare inputs / targets_template / forcings
# =============================================================================

def prepare_inputs(
    example_batch: xarray.Dataset,
    task_config,
    rollout_steps: int,
):
    """Split batch into inputs / targets_template / forcings.

    extract_inputs_targets_forcings internally:
      1. Shifts time axis by (target_duration - time[-1])
         so last input maps to t=0 and targets map to positive lead times
      2. Slices inputs  at (-input_duration, 0h]
      3. Slices targets at target_lead_times
      4. Computes derived forcings (sin/cos year/day progress, solar)

    Normalization is handled inside the model wrapper -- no manual scaling needed.
    """
    inputs, targets, forcings = data_utils.extract_inputs_targets_forcings(
        example_batch,
        target_lead_times=slice("6h", f"{rollout_steps * 6}h"),
        **dataclasses.asdict(task_config),
    )
    targets_template = targets * np.nan

    print(f"inputs           : {dict(inputs.sizes)}  vars={sorted(inputs.data_vars)}")
    print(f"targets_template : {dict(targets_template.sizes)}  vars={sorted(targets_template.data_vars)}")
    print(f"forcings         : {dict(forcings.sizes)}  vars={sorted(forcings.data_vars)}")

    # Inputs must be NaN-free -- NaN here = NaN predictions
    for var in inputs.data_vars:
        n = int(np.isnan(inputs[var].values).sum())
        if n > 0:
            raise ValueError(f"NaN in input '{var}': {n} values -- check batch construction")
    print("Inputs clean.")

    return inputs, targets_template, forcings
    
# =============================================================================
# 7.  STEP 7 -- Run the autoregressive rollout
# =============================================================================
def run_rollout(
    run_forward_jitted,
    inputs: xarray.Dataset,
    targets_template: xarray.Dataset,
    forcings: xarray.Dataset,
) -> xarray.Dataset:
    print("Running rollout ... (first step includes JAX JIT compile time)")
    # Match demo exactly: targets_template=eval_targets * np.nan passed inline
    predictions = rollout.chunked_prediction(
        run_forward_jitted,
        rng=jax.random.PRNGKey(0),
        inputs=inputs,
        targets_template=targets_template * np.nan,  # ensure NaN even if already NaN
        forcings=forcings,
    )
    print("Rollout complete.  Predictions:", dict(predictions.sizes))
    return predictions


# =============================================================================
# 8.  VISUALISATION
# =============================================================================

def _select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None,
) -> xarray.DataArray:
    da = data[variable]
    if "batch" in da.dims:
        da = da.isel(batch=0)
    if max_steps is not None and "time" in da.sizes:
        da = da.isel(time=range(0, min(max_steps, da.sizes["time"])))
    if level is not None and "level" in da.coords:
        da = da.sel(level=level)
    return da


def _scale(
    da: xarray.DataArray,
    center: Optional[float] = None,
    robust: bool = True,
) -> tuple:
    vmin = float(np.nanpercentile(da.values, 2 if robust else 0))
    vmax = float(np.nanpercentile(da.values, 98 if robust else 100))
    if center is not None:
        diff = max(vmax - center, center - vmin)
        vmin, vmax = center - diff, center + diff
    return da, mcolors.Normalize(vmin, vmax), ("RdBu_r" if center is not None else "viridis")


def plot_forecast(
    predictions: xarray.Dataset,
    init_time: datetime.datetime,
    variable: str = "2m_temperature",
    level: Optional[int] = None,
    step: int = 0,
    robust: bool = True,
    figsize: tuple = (12, 5),
) -> plt.Figure:
    """Plot a single forecast step as a global map.

    Parameters
    ----------
    step  : forecast step index (0 -> +6h, 1 -> +12h, ...)
    level : pressure level in hPa; required for 3-D variables
    """
    da       = _select(predictions, variable, level)
    da_step  = da.isel(time=step)
    lead_h   = (step + 1) * 6
    valid_dt = init_time + datetime.timedelta(hours=lead_h)

    title = variable
    if level is not None:
        title += f" @ {level} hPa"
    title += f"  |  +{lead_h}h  (valid {valid_dt.strftime('%Y-%m-%d %HZ')})"

    _, norm, cmap = _scale(da, robust=robust)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(
        da_step.values, norm=norm, cmap=cmap, origin="lower",
        extent=[-180, 180, -90, 90], aspect="auto",
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, orientation="vertical", shrink=0.75, pad=0.02)
    plt.tight_layout()
    return fig


def plot_multi_variable_snapshot(
    predictions: xarray.Dataset,
    init_time: datetime.datetime,
    variables: list[str],
    level: Optional[int] = None,
    step: int = 0,
    robust: bool = True,
    cols: int = 2,
    plot_size: float = 4,
) -> plt.Figure:
    """Side-by-side global maps for multiple variables at the same forecast step."""
    rows     = math.ceil(len(variables) / cols)
    lead_h   = (step + 1) * 6
    valid_dt = init_time + datetime.timedelta(hours=lead_h)

    fig = plt.figure(figsize=(plot_size * 2 * cols, plot_size * rows))
    fig.suptitle(
        f"GraphCast  |  +{lead_h}h  (valid {valid_dt.strftime('%Y-%m-%d %HZ')})",
        fontsize=13,
    )
    for i, var in enumerate(variables):
        da      = _select(predictions, var, level, max_steps=step + 1)
        da_step = da.isel(time=step)
        _, norm, cmap = _scale(da, robust=robust)
        ax = fig.add_subplot(rows, cols, i + 1)
        im = ax.imshow(
            da_step.values, norm=norm, cmap=cmap, origin="lower",
            extent=[-180, 180, -90, 90], aspect="auto",
        )
        ax.set_title(var if level is None else f"{var} @ {level} hPa", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, orientation="vertical", shrink=0.75, pad=0.02)

    fig.tight_layout()
    return fig


def animate_forecast(
    predictions: xarray.Dataset,
    init_time: datetime.datetime,
    variable: str = "2m_temperature",
    level: Optional[int] = None,
    robust: bool = True,
    interval_ms: int = 300,
) -> HTML:
    """Animate all forecast steps as an inline HTML widget (Jupyter).

    Usage:
        from IPython.display import display
        display(animate_forecast(predictions, init_time, "2m_temperature"))
    """
    da = _select(predictions, variable, level)
    _, norm, cmap = _scale(da, robust=robust)
    n_steps = da.sizes["time"]

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(
        da.isel(time=0).values, norm=norm, cmap=cmap,
        origin="lower", extent=[-180, 180, -90, 90], aspect="auto",
    )
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, orientation="vertical", shrink=0.75, pad=0.02)

    def _update(frame):
        lead_h   = (frame + 1) * 6
        valid_dt = init_time + datetime.timedelta(hours=lead_h)
        title = variable
        if level is not None:
            title += f" @ {level} hPa"
        title += f"  |  +{lead_h}h  (valid {valid_dt.strftime('%Y-%m-%d %HZ')})"
        ax.set_title(title, fontsize=12)
        im.set_data(da.isel(time=frame).values)

    ani = animation.FuncAnimation(fig, _update, frames=n_steps, interval=interval_ms)
    plt.close(fig)
    return HTML(ani.to_jshtml())


# =============================================================================
# 9.  TOP-LEVEL PIPELINE FUNCTION
# =============================================================================

def run_graphcast_inference(
    init_time: datetime.datetime,
    rollout_steps: int,
    ckpt_path: str | Path,
    stats_dir: str | Path,
    era5_dir: str | Path = "era5_data",
    use_era5_solar: bool = True,
    skip_download: bool = False,
) -> xarray.Dataset:
    """End-to-end GraphCast inference pipeline.

    Parameters
    ----------
    init_time     : forecast init time (00/06/12/18 UTC).
                    ERA5 will be fetched at init_time-12h and init_time-6h.
    rollout_steps : number of 6-hourly steps to predict (40 = 10 days).
    ckpt_path     : path to downloaded .npz checkpoint.
    stats_dir     : directory containing the three normalization .nc files.
    era5_dir      : directory to save / read ERA5 NetCDF files.
    use_era5_solar: if True, download ERA5 solar radiation for forcings.
                    if False, GraphCast computes it analytically.
    skip_download : set True to reuse existing ERA5 files in era5_dir.

    Returns
    -------
    predictions : xarray.Dataset
    """

    # 1. Download ERA5
    era5_dir = Path(era5_dir)
    
        # 1. Download ERA5 -- each component skipped automatically if file exists.
        #    Re-run safely at any time; only missing files are fetched.
    nc_paths = download_era5_inputs(
        init_time=init_time,
        rollout_steps=rollout_steps,
        output_dir=era5_dir,
        use_era5_solar=use_era5_solar,
    )


    # 2. Build xarray batch
    print("\n[STEP 2] Building xarray batch ...")
    example_batch = build_era5_xarray_batch(nc_paths, init_time, rollout_steps)
    
    # 3. Load checkpoint
    print("\n[STEP 3] Loading checkpoint ...")
    params, state, model_config, task_config = load_checkpoint(ckpt_path)

    # 4. Load normalization stats
    print("\n[STEP 4] Loading normalization statistics ...")
    diffs_stddev, mean, stddev = load_normalization_stats(stats_dir)

    # 5. Build jitted forward function
    print("\n[STEP 5] Building jitted forward function ...")
    run_forward_jitted = build_run_forward_jitted(
        model_config, task_config, params, state, diffs_stddev, mean, stddev
    )

    # 6. Prepare inputs / targets_template / forcings
    print("\n[STEP 6] Preparing inputs ...")
    inputs, targets_template, forcings = prepare_inputs(
        example_batch, task_config, rollout_steps
    )

    # 7. Run rollout
    print("\n[STEP 7] Running rollout ...")
    predictions = run_rollout(run_forward_jitted, inputs, targets_template, forcings)

    return predictions




In [4]:
CFG = {
    # -- Forecast settings ----------------------------------------------------
    "init_time":     datetime.datetime(2026, 1, 1, 0, 0),  # must be 00/06/12/18 UTC
    "rollout_steps": 40,                                    # 40 × 6h = 10 days

    # -- Paths ----------------------------------------------------------------
    "ckpt_path": (
        "/raid/apaudel/graphcast/params/"
        "GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - "
        "pressure levels 13 - mesh 2to6 - precipitation output only.npz"
    ),
    "stats_dir":  "/raid/apaudel/graphcast/stats/",
    "era5_dir":   "/home/apaudel/PROJECTS/graphcast/inference/era5_data/",
    "output_dir": "/home/apaudel/PROJECTS/graphcast/inference/outputs/",

    # -- ERA5 download options ------------------------------------------------
    "use_era5_solar": False,
    "skip_download":  False,

    # -- Plot settings --------------------------------------------------------
    # step index: 0=+6h, 1=+12h, 3=+24h, 7=+48h, 19=+120h (5d), 39=+240h (10d)
    "plots": [
        {"type": "single", "variable": "2m_temperature",         "level": None, "step": 3,  "filename": "t2m_24h.png"},
        {"type": "single", "variable": "mean_sea_level_pressure", "level": None, "step": 7,  "filename": "mslp_48h.png"},
        {"type": "single", "variable": "geopotential",            "level": 500,  "step": 7,  "filename": "z500_48h.png"},
        {"type": "single", "variable": "u_component_of_wind",     "level": 850,  "step": 19, "filename": "u850_5d.png"},
        {"type": "multi",  "step": 7,  "level": None, "filename": "multi_surface_48h.png",
         "variables": ["2m_temperature", "mean_sea_level_pressure",
                       "10m_u_component_of_wind", "10m_v_component_of_wind"]},
        {"type": "multi",  "step": 19, "level": 500,  "filename": "multi_upper_5d.png",
         "variables": ["geopotential", "temperature",
                       "u_component_of_wind", "v_component_of_wind"]},
    ],

    # -- Animation settings ---------------------------------------------------
    "animations": [
        {"variable": "2m_temperature",          "level": None, "filename": "anim_t2m.html"},
        {"variable": "mean_sea_level_pressure",  "level": None, "filename": "anim_mslp.html"},
        {"variable": "geopotential",             "level": 500,  "filename": "anim_z500.html"},
    ],
}

In [5]:
# # =============================================================================
# # 10.  CONFIG + EXAMPLE USAGE
# # =============================================================================

# def main(cfg: dict) -> xarray.Dataset:
#     """Run the full GraphCast inference pipeline from a config dict.

#     Parameters
#     ----------
#     cfg : dict
#         Pipeline configuration. See CFG above for all keys and descriptions.

#     Returns
#     -------
#     predictions : xarray.Dataset
#     """
#     output_dir = Path(cfg["output_dir"])
#     output_dir.mkdir(parents=True, exist_ok=True)

#     # Run inference
#     predictions = run_graphcast_inference(
#         init_time=cfg["init_time"],
#         rollout_steps=cfg["rollout_steps"],
#         ckpt_path=cfg["ckpt_path"],
#         stats_dir=cfg["stats_dir"],
#         era5_dir=cfg["era5_dir"],
#         use_era5_solar=cfg["use_era5_solar"],
#         skip_download=cfg["skip_download"],
#     )

#     # Generate plots
#     for p in cfg["plots"]:
#         if p["type"] == "single":
#             fig = plot_forecast(
#                 predictions, cfg["init_time"],
#                 variable=p["variable"],
#                 level=p["level"],
#                 step=p["step"],
#             )
#         else:  # multi
#             fig = plot_multi_variable_snapshot(
#                 predictions, cfg["init_time"],
#                 variables=p["variables"],
#                 level=p["level"],
#                 step=p["step"],
#             )
#         save_path = output_dir / p["filename"]
#         fig.savefig(save_path, dpi=150, bbox_inches="tight")
#         plt.close(fig)
#         print(f"Saved {save_path}")

#     # Generate animations
#     for a in cfg["animations"]:
#         html = animate_forecast(
#             predictions, cfg["init_time"],
#             variable=a["variable"],
#             level=a["level"],
#         )
#         save_path = output_dir / a["filename"]
#         save_path.write_text(html.data)
#         print(f"Saved {save_path}")

#     return predictions


# if __name__ == "__main__":
#     main(CFG)

In [6]:
# =============================================================================
# 10.  CONFIG + EXAMPLE USAGE
# =============================================================================

cfg = CFG
output_dir = Path(cfg["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# Run inference
predictions = run_graphcast_inference(
    init_time=cfg["init_time"],
    rollout_steps=cfg["rollout_steps"],
    ckpt_path=cfg["ckpt_path"],
    stats_dir=cfg["stats_dir"],
    era5_dir=cfg["era5_dir"],
    use_era5_solar=cfg["use_era5_solar"],
    # skip_download=cfg["skip_download"],
)



[SKIP] Pressure-level variables -> /home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_pressure_levels.nc
[SKIP] Surface variables -> /home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_surface.nc
[SKIP] Static variables -> /home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_static.nc
ERA5 download complete.

[STEP 2] Building xarray batch ...
Batch: {'batch': 1, 'time': 42, 'level': 13, 'lat': 721, 'lon': 1440}
  time : [              0  21600000000000  43200000000000  64800000000000
  86400000000000 108000000000000 129600000000000 151200000000000
 172800000000000 194400000000000 216000000000000 237600000000000
 259200000000000 280800000000000 302400000000000 324000000000000
 345600000000000 367200000000000 388800000000000 410400000000000
 432000000000000 453600000000000 475200000000000 496800000000000
 518400000000000 540000000000000 561600000000000 583200000000000
 604800000000000 626400000000000 648000000000000 669600000000000
 691200000000000 712800000000000 7344

/home/apaudel/PROJECTS/graphcast/graphcast_fork_ap/graphcast/rollout.py:295: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  num_target_steps = targets_template.dims["time"]
/home/apaudel/PROJECTS/graphcast/graphcast_fork_ap/graphcast/autoregressive.py:115: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  num_inputs = inputs.dims['time']
/home/apaudel/PROJECTS/graphcast/graphcast_fork_ap/graphcast/rollout.py:400: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping

Rollout complete.  Predictions: {'time': 40, 'batch': 1, 'lat': 721, 'lon': 1440, 'level': 13}


In [7]:
# Generate plots
for p in cfg["plots"]:
    if p["type"] == "single":
        fig = plot_forecast(
            predictions, cfg["init_time"],
            variable=p["variable"],
            level=p["level"],
            step=p["step"],
        )
    else:  # multi
        fig = plot_multi_variable_snapshot(
            predictions, cfg["init_time"],
            variables=p["variables"],
            level=p["level"],
            step=p["step"],
        )
    save_path = output_dir / p["filename"]
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {save_path}")

# Generate animations
for a in cfg["animations"]:
    html = animate_forecast(
        predictions, cfg["init_time"],
        variable=a["variable"],
        level=a["level"],
    )
    save_path = output_dir / a["filename"]
    save_path.write_text(html.data)
    print(f"Saved {save_path}")



Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/t2m_24h.png
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/mslp_48h.png
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/z500_48h.png
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/u850_5d.png
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/multi_surface_48h.png
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/multi_upper_5d.png
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/anim_t2m.html
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/anim_mslp.html
Saved /home/apaudel/PROJECTS/graphcast/inference/outputs/anim_z500.html


In [8]:
predictions

<xarray.Dataset> Size: 14GB
Dimensions:                  (time: 40, batch: 1, lat: 721, lon: 1440, level: 13)
Coordinates:
  * lat                      (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * lon                      (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
  * time                     (time) timedelta64[ns] 320B 0 days 06:00:00 ... ...
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 166MB 1.342 ... ...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 166MB -0.1805 .....
    2m_temperature           (time, batch, lat, lon) float32 166MB 245.3 ... ...
    geopotential             (time, batch, level, lat, lon) float32 2GB 2.031...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 166MB 1.017e+05 ...
    specific_humidity        (time, batch, level, lat, lon) float32 2GB 2.956...
    temperature              (time, batch, level, lat, lon) float32 2GB 236.4...
    total_precipitation_6hr  (time, batch, lat, lon) float32 166MB -2.489e-05...
    u_component_of_wind      (time, batch, level, lat, lon) float32 2GB 0.590...
    v_component_of_wind      (time, batch, level, lat, lon) float32 2GB 0.103...
    vertical_velocity        (time, batch, level, lat, lon) float32 2GB -0.00...

In [9]:
predictions['time'].values

array([ 21600000000000,  43200000000000,  64800000000000,  86400000000000,
       108000000000000, 129600000000000, 151200000000000, 172800000000000,
       194400000000000, 216000000000000, 237600000000000, 259200000000000,
       280800000000000, 302400000000000, 324000000000000, 345600000000000,
       367200000000000, 388800000000000, 410400000000000, 432000000000000,
       453600000000000, 475200000000000, 496800000000000, 518400000000000,
       540000000000000, 561600000000000, 583200000000000, 604800000000000,
       626400000000000, 648000000000000, 669600000000000, 691200000000000,
       712800000000000, 734400000000000, 756000000000000, 777600000000000,
       799200000000000, 820800000000000, 842400000000000, 864000000000000],
      dtype='timedelta64[ns]')

In [10]:
print(predictions)

<xarray.Dataset> Size: 14GB
Dimensions:                  (time: 40, batch: 1, lat: 721, lon: 1440, level: 13)
Coordinates:
  * lat                      (lat) float32 3kB -90.0 -89.75 -89.5 ... 89.75 90.0
  * lon                      (lon) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
  * time                     (time) timedelta64[ns] 320B 0 days 06:00:00 ... ...
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 166MB 1.342 ... ...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 166MB -0.1805 .....
    2m_temperature           (time, batch, lat, lon) float32 166MB 245.3 ... ...
    geopotential             (time, batch, level, lat, lon) float32 2GB 2.031...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 166MB 1.017e+05 ...
    specific_humidity        (time, batch, level, lat, lon) float32 2GB 2.956...
    temperature     

In [11]:
def debug_batch_nans(nc_paths, init_time):
    """Call this instead of build_era5_xarray_batch to find where NaN enters."""
    
    print("\n=== RAW FILE NaN CHECK ===")
    
    # Check raw files BEFORE any processing
    for label, path in nc_paths.items():
        ds = xarray.load_dataset(path).compute()
        print(f"\n  [{label}] raw vars and NaN counts:")
        print(f"    coords: {list(ds.coords)}")
        print(f"    dims  : {dict(ds.sizes)}")
        for var in ds.data_vars:
            n_nan = int(np.isnan(ds[var].values.astype(float)).sum())
            n_tot = ds[var].values.size
            print(f"    {var:30s}  dtype={ds[var].dtype}  NaN={n_nan}/{n_tot}")

In [12]:

  # pressure_levels : /home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_pressure_levels.nc

  # surface         : 

  # static          : 

  # solar           : 

init_time=datetime.datetime(2026, 1, 1, 0, 0)

ncpath = {"pressure": "/home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_pressure_levels.nc",
         "surface": "/home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_surface.nc",
         "static": "/home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_static.nc",
         "solar": "/home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_solar_forcing.nc"}


In [13]:
debug_batch_nans(ncpath, init_time)



=== RAW FILE NaN CHECK ===

  [pressure] raw vars and NaN counts:
    coords: ['number', 'valid_time', 'pressure_level', 'latitude', 'longitude', 'expver']
    dims  : {'valid_time': 2, 'pressure_level': 13, 'latitude': 721, 'longitude': 1440}
    z                               dtype=float32  NaN=0/26994240
    q                               dtype=float32  NaN=0/26994240
    t                               dtype=float32  NaN=0/26994240
    u                               dtype=float32  NaN=0/26994240
    v                               dtype=float32  NaN=0/26994240
    w                               dtype=float32  NaN=0/26994240

  [surface] raw vars and NaN counts:
    coords: ['number', 'valid_time', 'latitude', 'longitude', 'expver']
    dims  : {'valid_time': 2, 'latitude': 721, 'longitude': 1440}
    t2m                             dtype=float32  NaN=0/2076480
    msl                             dtype=float32  NaN=0/2076480
    u10                             dtype=float32  Na

FileNotFoundError: [Errno 2] No such file or directory: '/home/apaudel/PROJECTS/graphcast/inference/era5_data/era5_solar_forcing.nc'